# Self evalutation test 1

# Self evalutation test 1

The exercises that follows are to be done and sent, via e-mail, to:
- dario.tamascelli@uni-ulm.de
-thibaut.lacroix@uni-ulm.de
within Tuesday,May, 15-th, 2026

The provided solutions will checked and marked, as to provide you with an assessment of your current level of understanding and preparation. 

## Exercise 1

1. Prepare a system of $n=10$ spin $1/2$ particles.

2. Create an MPS corresponding to an initial state $\ket{\psi_0}$ with all spins downs.

3. Operate on $\ket{\psi_0}$ with suitable operators to create the GHZ state

$$ 
\frac{\ket{\uparrow_1 \uparrow_2,\ldots, \uparrow_n} +\ket{\downarrow_1 \downarrow_2,\ldots, \downarrow_n}}{\sqrt{2}}
$$
4. Print the bond dimensions of the resulting state.

5. Compute the von Neumann entropy across all bonds $j \rightarrow (j+1)$ for $j=1,2,\ldots, n-1$.

6. Measure, for all sites, the expectation value of the Pauli operators $\sigma_z$, $\sigma_x$ and $\sigma_y$. Show the results by plotting the so obtained expectations in 3 plots, each showing the number of the site in abscissae and the expectation value of $\sigma_k \; k=\{x,y,z\}$ in oridinates.  

In [1]:
using Pkg
Pkg.activate("../")
using ITensors
using ITensorMPS
using Plots
using LinearAlgebra

  Activating project at `c:\Users\mariu\GithubCloneTN\TN_Notebooks`


In [ ]:
nn = 10
system = siteinds("S=1/2",nn)

psi0= MPS(ComplexF64, system, ["Dn" for j in 1:nn])

In [ ]:
# Create operators that change the spin of every site

asc = MPO(system, "S+")

In [ ]:
op = asc + MPO(system, "Id")
ghz = noprime(op*psi0)
normalize!(ghz)


In [ ]:
# check the created ghz state
expect(ghz, "Z")

In [ ]:
# explicitly print the bond dimensions

for i in 1:nn-1
    println(dim(commonind(ghz[i], ghz[i+1])))
end

In [ ]:
# for each bond, make an svd, and calculate the entropy from s_i. use the bond to the right for the cut.

entropies = []
for i in 1:nn-1
    orthogonalize!(ghz, i)
    uu, s, vv = svd(ghz[i], commonind(ghz[i], ghz[i+1]))
    entr = 0
    s = matrix(s, inds(s))
    for j in 1:size(s)[1]
        if s[j,j] >= 10e-10
            entr -= s[j,j]^2*log(s[j,j]^2)
        end
    end
    push!(entropies, entr)
end


In [ ]:
entropies

In [ ]:
# write functions, that measure with sigma_XYZ_siteJ

function measureX(state, j)
    opsum = OpSum()
    opsum += "X", j
    op = MPO(opsum, system)
    return inner(state', op, state)
end

function measureY(state, j)
    opsum = OpSum()
    opsum += "Y", j
    op = MPO(opsum, system)
    return inner(state', op, state)
end

function measureZ(state, j)
    opsum = OpSum()
    opsum += "Z", j
    op = MPO(opsum, system)
    return inner(state', op, state)
end

In [ ]:
# create the separate plots with the expectation value for X, Y, Z over the siteindex

expX = [real(measureX(ghz, i)) for i in 1:nn]
plot(1:nn, expX)



In [ ]:
expY = [real(measureY(ghz, i)) for i in 1:nn]
plot(1:nn, expY)

In [ ]:
expZ = [real(measureZ(ghz, i)) for i in 1:nn]
plot(1:nn, expZ)

## Exercise 2

Once defined a system of $n=10$ spin $1/2$ particles:

1. Define the MPS of the initial "all up" state $\ket{psi_0} = \ket{\uparrow_1 \uparrow_2,\ldots, \uparrow_n}$.
2. Prepare the MPO corresponding to the operator (measurement):

$$
N_z = \sum_{j=1}^n \frac{1_j + \sigma_j^z}{2};
$$

apply the measurement $\bra{\psi_0} N_z \ket{\psi_0}$ to the state $\ket{\psi_0}$ and print the result.

3. Compute the von Neumann entropy across all bonds $j \rightarrow (j+1)$ for $j=1,2,\ldots, n-1$.

4. Prepare the MPO corresponding to the operator:
$$
M = \sum_{j=1}^{n-1} \frac{1_j + \sigma_j^z}{\sqrt{2}} \otimes \sigma_{j+1}^x
$$

5. Apply the MPO $M$ to the initial  $\ket{\psi_0} = \ket{\uparrow_1 \uparrow_2,\ldots, \uparrow_n}$.

6. Print the result of the measurement of $N_z$ on $\ket{\phi} = M \ket{\psi_0}$. 

7. Compute and the new  the von Neumann entropy across all bonds $j \rightarrow (j+1)$ for $j=1,2,\ldots, n-1$.

8. Measure, print and plot the values $\bra{\phi}\sigma_j^z\ket{\phi}$ for $j=1,2,\ldots,n$.




In [ ]:
# use the system and nn of the prev exercise

psi0= MPS(ComplexF64, system, ["Up" for j in 1:nn])

In [ ]:
# define the operator 

opsum = OpSum()
for j = 1:nn
    opsum += 1/2, "Id", j
    opsum += 1/2, "Z", j
end
N_Z = MPO(opsum, system)



In [ ]:
# make the measurement:

inner(psi0', N_Z, psi0)

In [ ]:
# Calculate a vector of the entropies over the bonds, similar to ex. 1

entropies = []
for i in 1:nn-1
    orthogonalize!(psi0, i)
    uu, s, vv = svd(psi0[i], commonind(psi0[i], psi0[i+1]))
    entr = 0
    s = matrix(s, inds(s))
    for j in 1:size(s)[1]
        if s[j,j] >= 10e-10
            entr -= s[j,j]^2*log(s[j,j]^2)
        end
    end
    push!(entropies, entr)
end

In [ ]:
entropies

In [ ]:
# Define the new op M

opsum = OpSum()
for j = 1:nn-1
    opsum += 1/sqrt(2), "Id", j, "X", j+1
    opsum += 1/sqrt(2), "Z", j, "X", j+1
end
M = MPO(opsum, system)

In [ ]:
# apply to the state psi0 and measure

phi = noprime(M*psi0)
normalize!(phi)

inner(phi', N_Z, phi)

In [ ]:
# calculate the entropies for phi in the same way

entropies = []
for i in 1:nn-1
    orthogonalize!(phi, i)
    uu, s, vv = svd(phi[i], commonind(phi[i], phi[i+1]))
    entr = 0
    s = matrix(s, inds(s))
    for j in 1:size(s)[1]
        if s[j,j] >= 10e-10
            entr -= s[j,j]^2*log(s[j,j]^2)
        end
    end
    push!(entropies, entr)
end

In [ ]:
plot(1:nn-1, entropies)

In [ ]:
# we want to plot the measurements of sigma_z on phi at site j. Use the function measureZ from ex. 1. 

expZ = [real(measureZ(phi, i)) for i in 1:nn]
plot(1:nn, expZ)

## Exercise 3

Let us consider the Heisenberg Hamiltonian

$$
H_\text{heis} = -J \sum_{j=1}^{n-1} \sigma_j^- \sigma_{j+1}^+ +\frac{1}{2} \sigma_j^z \sigma_{j+1}^z.
$$

for a system of $n=20$ spin $1/2$ particles starting from the state:
$$
\ket{\psi_0} = \ket{\uparrow_1 \downarrow_2 \uparrow_3 \downarrow_4\ldots \uparrow_{19} \downarrow_{20}},
$$
i.e. with couple of neighboring spins aligned/anti-aligned to the $z$ direction. 

1. Compute the energy of the initial state $\ket{\psi_0}$. 

2. Define the MPO of $U(\delta t) = \exp\left (-i \delta t H_\text{heis}\right )$ for $\delta t=0.01$.

3. Determine the evolution $\ket{\psi_{k  \delta t}} = U(k  \delta t)\ket{\psi_0}$ for $k=1,2,\ldots,200$; at each time-step (including the initial time $t= 0 \delta t$) measure and record:
    - the energy of the state
    - the average number of spins up
    - the von Neumann entropy across all bonds $j \rightarrow (j+1)$
    - the norm of the state

A the end of the simulation plot:

- the energy as a funcition of (discrete) times
- the norm as a function of time
- the average number of spins up
- an animated plot with the von Neumann entropy across the bonds, for all evolution steps.

In [ ]:
# define a new syystem and site numer nn

nn = 20
system = siteinds("S=1/2", nn)

In [ ]:
# define the state with alternating up down 

psi0 = MPS(ComplexF64, system, [(j%2 == 1) ? "Up" : "Dn" for j in 1:nn])

In [ ]:
# define the Heisenberg Hamiltonian

J = 1
opsum = OpSum()
for j in 1:nn-1
    opsum += -1*J, "S-", j, "S+", j+1
    opsum += -1*J/2, "Z", j, "Z", j+1
end
Ham = MPO(opsum, system)


In [ ]:
# Calculate the initial enregy with this hamiltonian

inner(psi0', Ham, psi0)

In [ ]:
# define the evolution operator for a small time step delta_t. Use the taylor approximation of a prev. exercise.


deltat = 0.01
nsteps = Int(round(2/deltat))+1
hdeltat = deltat * Ham
#build an MPO with the operator "Id" on each site
theOne = MPO(system,"Id")
factor(k) = (-1im)^k / factorial(k)
#As not to have all the indices of hdeltat contracted with each other, we prime the second one and then we contract the first index of the second one with the first index of the first one, and then we lower the second prime level to first prime level.
h2 = mapprime(prime(hdeltat)*hdeltat,2 =>1)
h3 = mapprime(prime(hdeltat) * h2,2 =>1)
terms = [theOne, factor(1)*hdeltat, factor(2)*h2, factor(3)*h3]
appo = terms[1];
for t in terms[2:end]
    appo = t + appo
end
h3thOrder = appo

In [ ]:
# create a vector of the evolved states by repeatedly applying the above MPO. 

evolved = Vector{MPS}()
push!(evolved, psi0) # the first element is the un evolved state for t=0
for j=2:nsteps
    push!(evolved, noprime(h3thOrder * evolved[end]))
end


In [ ]:
# we have the evolved states, and can calculate various quantities over time:

# energy

energy = []
for state in evolved
    push!(energy, inner(state', Ham, state))
end

# avg number spins up
# use the function expect.

expect(psi0, "Sz")


In [ ]:
# result: A vector with the eigenvalues of Sz. 

In [ ]:
avgUp = []
for state in evolved
    eig = ones(nn)+2*expect(state, "Sz") # add one, so that we have up -> 1, dn -> 0
    avg = sum(eig)/nn
    push!(avgUp, avg)
end

In [ ]:
# norm

norms = []

for state in evolved
    push!(norms, inner(state', state))
end

In [ ]:
times = 0:deltat:2

In [ ]:
# we have now these 3 quantities, plot them over t or times

In [ ]:
plot(times, real(norms))

In [ ]:
plot(times, real(energy))

In [ ]:
plot(times, real(avgUp))

In [ ]:
# make an animated plot of the von neumann entropies over time.
# create a function, that returns the entropy over the bond j -> j+1

function entrop(phi, i)
        orthogonalize!(phi, i)
        uu, s, vv = svd(phi[i], commonind(phi[i], phi[i+1]))
        entr = 0
        s = matrix(s, inds(s))
        for j in 1:size(s)[1]
            if s[j,j] >= 10e-10
                entr -= s[j,j]^2*log(s[j,j]^2)
            end
        end
        return entr
end

In [ ]:
anim = @animate for i=1:nsteps
    fig = plot(real([entrop(evolved[i],j) for j in 1:nn-1]), seriestype=:scatter, label="Entropies over time and index j -> j+1")
    plot!(fig, real([entrop(evolved[i],j) for j in 1:nn-1]))
end;
gif(anim, "evolution.gif", fps=10)

In [ ]:
# Apparantly, the norm is stable, but the energy and the entropy explode, maybe we should use smaller time steps or a higher order approximation for the hamiltonian.